# Clothes segmentation: why our model beats every other model

Scores five models on the same 986 CCP photos that none of them was trained on, then explains, **in one cell**, why each of the others is below our final model:

- the garments where it loses most, and which class those pixels went to;
- the kind of error: outlines, small garments, garments missed or invented;
- the six photos where it falls furthest behind ours.

For YOLO26x-sem the explanation also covers ATR validation, where it was trained on the same photos as our SegFormer-B3. That is the cleanest test of the architecture.

| model | trained by | on |
|---|---|---|
| our final model, SegFormer-B3, 30 epochs | us | ATR |
| our SegFormer-B3, 12 epochs | us | ATR |
| YOLO26x-sem, 12 epochs (trained with `colab_model_compare.ipynb`) | us | ATR |
| SCHP-ATR | others | ATR |
| FASHN Human Parser (SegFormer-B4) | others | their own data |

A **T4 GPU is enough**. About 40 minutes. Run the cells from top to bottom; cell 3 asks for `clothseg_code.zip` only if Drive has an older code version.

## 1. Settings

In [ ]:
REPO_URL = ""

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"
CODE_DIR = "/content/clothes-segmentation"
CODE_VERSION = "19"
HEIGHT, WIDTH = 576, 384
SESSION_BUDGET_MIN = 600   # used by a helper in cell 2; nothing is trained here

IMGSZ = 576
YOLO_BEST = f"{DRIVE_ROOT}/runs_yolo/yolo26x_sem_e12/weights/best.pt"
ATR_CACHE = f"/content/data/atr_{HEIGHT}x{WIDTH}"

## 2. Mount Drive and show the environment

In [ ]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

## 3. Get the code

In [ ]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

## 4. Install requirements and Ultralytics, prepare ATR

In [ ]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}
!pip install -q "ultralytics==8.4.160"
import ultralytics
print("ultralytics", ultralytics.__version__)
try:
    from ultralytics.engine.results import SemanticMask   # exists only in versions with semantic segmentation
except ImportError:
    raise AssertionError("This ultralytics version has no semantic segmentation; it needs 8.4.52 or newer.")
print("semantic segmentation available")

## 5. Download CCP, read its labels, remove photos that are also in ATR

In [ ]:
import os, subprocess

CCP_DIR = "/content/ccp"
if not os.path.isdir(f"{CCP_DIR}/annotations/pixel-level"):
    print("downloading the Clothing Co-Parsing dataset from GitHub (about 130 MB) ...", flush=True)
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bearpaw/clothing-co-parsing.git", CCP_DIR],
                       capture_output=True, text=True)
    assert r.returncode == 0, "Download failed. Send me this message: " + r.stderr[-2000:]
n_ann = len([f for f in os.listdir(f"{CCP_DIR}/annotations/pixel-level") if f.endswith(".mat")])
n_img = len([f for f in os.listdir(f"{CCP_DIR}/photos") if f.endswith(".jpg")])
print(f"CCP ready: {n_img} photos, {n_ann} with pixel labels (only these are used)")

import glob, os, sys
import numpy as np
import pandas as pd
import scipy.io as sio
from PIL import Image

sys.path.insert(0, f"{CODE_DIR}/src")
from clothseg.data.preprocess import _dhash
from clothseg.inference import load_image

# The classes all models are compared on. CCP has one "skin" class, so face, arms and legs
# of the models are merged into "skin" before comparing.
EVAL = ["background", "hat", "hair", "sunglasses", "upper-clothes", "skirt", "pants", "dress", "belt", "shoes", "skin", "bag", "scarf"]
GARMENTS = ["hat", "sunglasses", "upper-clothes", "skirt", "pants", "dress", "belt", "shoes", "bag", "scarf"]
# CCP label -> comparison class. Mapping fixed before any result (PLAN.md section 41); labels
# without a clear match are ignored (left out of the scores), not counted as errors.
CCP_TO_EVAL = {
    "null": "background", "hat": "hat", "hair": "hair", "sunglasses": "sunglasses", "skin": "skin",
    "blazer": "upper-clothes", "blouse": "upper-clothes", "cape": "upper-clothes", "cardigan": "upper-clothes",
    "coat": "upper-clothes", "hoodie": "upper-clothes", "jacket": "upper-clothes", "shirt": "upper-clothes",
    "sweater": "upper-clothes", "sweatshirt": "upper-clothes", "t-shirt": "upper-clothes", "top": "upper-clothes",
    "vest": "upper-clothes",
    "skirt": "skirt", "dress": "dress", "belt": "belt", "scarf": "scarf", "bag": "bag", "purse": "bag",
    "jeans": "pants", "leggings": "pants", "pants": "pants", "shorts": "pants",
    "boots": "shoes", "clogs": "shoes", "flats": "shoes", "heels": "shoes", "loafers": "shoes", "pumps": "shoes",
    "sandals": "shoes", "shoes": "shoes", "sneakers": "shoes", "wedges": "shoes",
}
CCP_IGNORED = {"accessories", "bodysuit", "bra", "bracelet", "earrings", "glasses", "gloves", "intimate", "jumper",
               "necklace", "panties", "ring", "romper", "socks", "stockings", "suit", "swimwear", "tie", "tights",
               "wallet", "watch"}
OVERLAP_MAX_BITS = 4   # a CCP photo whose hash differs from an ATR photo's in at most 4 of 64 bits counts as the same photo

names = [str(a[0]) if a.size else "" for a in sio.loadmat(f"{CCP_DIR}/label_list.mat")["label_list"].ravel()]
unknown = [n for n in names if n not in CCP_TO_EVAL and n not in CCP_IGNORED]
assert not unknown, f"CCP labels without a rule: {unknown}"
ccp_lut = np.full(256, 255, np.uint8)
for i, n in enumerate(names):
    if n in CCP_TO_EVAL:
        ccp_lut[i] = EVAL.index(CCP_TO_EVAL[n])

# Hashes of every ATR photo, computed the same way during preprocessing (on the original photo).
meta = pd.read_csv(f"{ATR_CACHE}/meta.csv", dtype={"dhash": str})
atr_hash = np.array([int(h, 16) for h in meta["dhash"]], dtype=np.uint64)

samples, skipped, ignored_px, total_px = [], [], 0, 0
for path in sorted(glob.glob(f"{CCP_DIR}/annotations/pixel-level/*.mat")):
    sid = os.path.basename(path)[:-4]
    raw = sio.loadmat(path)["groundtruth"]
    photo = f"{CCP_DIR}/photos/{sid}.jpg"
    if not os.path.exists(photo):
        skipped.append((sid, "no photo"))
        continue
    image = load_image(photo)
    if image.shape[:2] != raw.shape:
        skipped.append((sid, f"photo {image.shape[:2]} and labels {raw.shape} differ in size"))
        continue
    gt = ccp_lut[raw]
    ignored_px += int((gt == 255).sum())
    total_px += gt.size
    h = np.uint64(int(_dhash(Image.fromarray(image)), 16))
    bits = int(np.unpackbits((atr_hash ^ h).view(np.uint8)).reshape(-1, 64).sum(1).min())
    samples.append({"id": sid, "photo": photo, "gt": gt, "bits_to_nearest_atr": bits})

dist = np.array([s["bits_to_nearest_atr"] for s in samples])
clean = [s for s in samples if s["bits_to_nearest_atr"] > OVERLAP_MAX_BITS]
print(f"labelled photos read: {len(samples)}; skipped: {len(skipped)} {skipped[:3]}")
print(f"pixels with an ignored CCP label: {100 * ignored_px / max(1, total_px):.2f}%")
print(f"overlap with ATR (64-bit image hash): identical {int((dist == 0).sum())}, within {OVERLAP_MAX_BITS} bits {int((dist <= OVERLAP_MAX_BITS).sum())}, "
      f"within 8 bits {int((dist <= 8).sum())}")
print(f"photos used for the comparison (no overlap with ATR): {len(clean)}")
present = {c: sum(int((s["gt"] == k).any()) for s in clean) for k, c in enumerate(EVAL)}
print("photos containing each class:", ", ".join(f"{c} {n}" for c, n in present.items()))

## 6. Score all five models on CCP

In [ ]:
import time
import torch
from clothseg.external import FashnParser, SchpAtr, YoloSemSegmenter
from clothseg.inference import Segmenter
from clothseg.labels import class_names

assert os.path.exists(YOLO_BEST), f"No YOLO weights at {YOLO_BEST}. Train it first with notebooks/colab_model_compare.ipynb."
K = len(EVAL)
NAMES15 = class_names(15)


def lut_from(names, rule):
    lut = np.full(256, 255, np.uint8)
    for i, n in enumerate(names):
        lut[i] = EVAL.index(rule(n))
    return lut


def rule15(n):
    return "skin" if n in ("face", "arms", "legs") else n


# FASHN's own classes: its body parts become skin; it has no shoes class ("feet" is a body class),
# so shoes are left out of its comparison; jewelry has no comparison class and counts as background.
FASHN_RULE = {"background": "background", "face": "skin", "hair": "hair", "top": "upper-clothes", "dress": "dress",
              "skirt": "skirt", "pants": "pants", "belt": "belt", "bag": "bag", "hat": "hat", "scarf": "scarf",
              "glasses": "sunglasses", "arms": "skin", "hands": "skin", "legs": "skin", "feet": "skin", "torso": "skin",
              "jewelry": "background"}

MODELS = {
    "our final (B3, 30 ep)": (lambda: Segmenter(f"{DRIVE_ROOT}/runs/final_segformer_b3_e30/checkpoints/best.pt"), lambda m: lut_from(NAMES15, rule15)),
    "our B3, 12 ep": (lambda: Segmenter(f"{DRIVE_ROOT}/runs/r1_segformer_b3/checkpoints/best.pt"), lambda m: lut_from(NAMES15, rule15)),
    "YOLO26x-sem, 12 ep": (lambda: YoloSemSegmenter(YOLO_BEST, imgsz=IMGSZ), lambda m: lut_from(NAMES15, rule15)),
    "SCHP-ATR": (lambda: SchpAtr(), lambda m: lut_from(NAMES15, rule15)),
    "FASHN parser": (lambda: FashnParser(), lambda m: lut_from(m.names, FASHN_RULE.__getitem__)),
}
SHOES = EVAL.index("shoes")
G10 = [EVAL.index(c) for c in GARMENTS]
G9 = [k for k in G10 if k != SHOES]


def ious(cm):
    inter = np.diag(cm).astype(float)
    union = cm.sum(0) + cm.sum(1) - np.diag(cm)
    return np.where(union > 0, inter / np.maximum(union, 1), np.nan)


from clothseg.diagnose import Diagnosis

ccp_rows, ccp_preds = {}, {name: {} for name in MODELS}
diags = {name: Diagnosis(EVAL, GARMENTS) for name in MODELS}   # where each model's errors are (cell 13)
show_ids = {clean[i]["id"] for i in np.random.default_rng(0).permutation(len(clean))[:6]}
for name, (make, make_lut) in MODELS.items():
    print("scoring", name, "on", len(clean), "CCP photos ...", flush=True)
    model = make()
    lut = make_lut(model)
    cm10, cm9 = np.zeros((K, K), np.int64), np.zeros((K, K), np.int64)
    t0 = time.time()
    for s in clean:
        image = load_image(s["photo"])
        p = lut[model.predict(image)]
        gt = s["gt"]
        v = gt != 255
        cm10 += np.bincount(gt[v].astype(np.int64) * K + p[v], minlength=K * K).reshape(K, K)
        diags[name].update(gt, p, s["id"])
        v9 = v & (gt != SHOES)     # shoes left out for everyone in the 9-garment score
        cm9 += np.bincount(gt[v9].astype(np.int64) * K + p[v9], minlength=K * K).reshape(K, K)
        if s["id"] in show_ids:
            ccp_preds[name][s["id"]] = p
    i10, i9 = ious(cm10), ious(cm9)
    ccp_rows[name] = {"garment mIoU (10)": np.nanmean(i10[G10]) if name != "FASHN parser" else np.nan,
                      "garment mIoU without shoes (9)": np.nanmean(i9[G9]),
                      "mIoU (13)": np.nanmean(i10) if name != "FASHN parser" else np.nan,
                      **{EVAL[k]: i10[k] for k in range(1, K)},
                      "ms per photo": 1000 * (time.time() - t0) / len(clean)}
    if name == "FASHN parser":
        ccp_rows[name]["shoes"] = np.nan
    del model
    torch.cuda.empty_cache()

ccp_table = pd.DataFrame(ccp_rows)
print()
print(f"CCP, {len(clean)} photos none of the models was trained on, original size, no flip averaging")
print("(FASHN has no shoes class, so its 10-garment and 13-class scores are not comparable: use the 9-garment score)")
print(ccp_table.round(4).to_string())
ccp_table.to_csv(f"{DRIVE_ROOT}/reports/compare_models_ccp.csv")
ours = ccp_table["our final (B3, 30 ep)"]
print()
for name in ccp_table.columns[1:]:
    other = ccp_table[name]
    key = "garment mIoU without shoes (9)"
    gain = other[key] - ours[key]
    worst = min(other[c] - ours[c] for c in ("upper-clothes", "pants", "skirt", "dress"))
    verdict = "BETTER than ours by the rule" if gain >= 0.005 and worst >= -0.01 else "not better than ours by the rule"
    print(f"{name}: {key} {100 * gain:+.2f} points against our final model; largest core-garment drop {100 * worst:+.2f} -> {verdict}")

## 7. Why our model beats every other model

In [ ]:
import pickle, re, sys
import matplotlib.pyplot as plt
from PIL import Image
sys.path.insert(0, f"{CODE_DIR}/src")
from clothseg.data.splits import load_splits
from clothseg.diagnose import Diagnosis, explain
from clothseg.external import score_atr
from clothseg.labels import CLASSES_15, GARMENT_NAMES, build_lut, palette

pd.set_option("display.width", 250)
pickle.dump(diags, open(f"{DRIVE_ROOT}/reports/why_ccp.pkl", "wb"))
OURS = "our final (B3, 30 ep)"
PAL = np.array([[0, 0, 0], [128, 0, 0], [255, 0, 0], [170, 0, 51], [255, 85, 0], [0, 128, 0], [0, 85, 85], [0, 0, 85],
                [85, 51, 0], [255, 255, 0], [85, 255, 170], [0, 119, 221], [52, 86, 128]], np.uint8)
PAL_IGN = np.vstack([PAL, np.full((256 - len(PAL), 3), 255, np.uint8)])
by_id = {s["id"]: s for s in clean}
ours_model = MODELS[OURS][0]()
ours_lut = MODELS[OURS][1](ours_model)


def show(rows, title, out):
    """rows: list of lists of (title, picture); one figure, saved to Drive."""
    fig, axes = plt.subplots(len(rows), len(rows[0]), figsize=(2.5 * len(rows[0]), 4.2 * len(rows)), squeeze=False)
    for r, pics in enumerate(rows):
        for c, (t, pic) in enumerate(pics):
            axes[r, c].imshow(pic)
            axes[r, c].axis("off")
            axes[r, c].set_title(t, fontsize=9)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(out, dpi=90, bbox_inches="tight")
    plt.show()


def yolo_on_atr():
    """YOLO and SegFormer-B3 trained on the same ATR photos for 12 epochs: the cleanest test of the architecture."""
    val_ids = load_splits(ATR_CACHE)["val"]
    g15 = [n for n in CLASSES_15 if n in GARMENT_NAMES]
    makers = {"YOLO26x-sem": MODELS["YOLO26x-sem, 12 ep"][0], "SegFormer-B3": MODELS["our B3, 12 ep"][0]}
    d = {}
    for n, make in makers.items():
        print(f"   diagnosing {n} on {len(val_ids)} ATR validation photos ...", flush=True)
        m = make()
        d[n] = Diagnosis(CLASSES_15, g15)
        score_atr(m.predict, ATR_CACHE, val_ids, diagnosis=d[n])
        del m
        torch.cuda.empty_cache()
    pickle.dump(d, open(f"{DRIVE_ROOT}/reports/why_atr_val.pkl", "wb"))
    print("   On ATR validation, both trained on the same photos for 12 epochs:")
    for line in explain(d["YOLO26x-sem"], d["SegFormer-B3"], "YOLO", "B3", top=6):
        print("    -", line)
    models = {n: make() for n, make in makers.items()}
    pal15, lut15, rows = palette(15), build_lut(15), []
    for gap, k in d["YOLO26x-sem"].worst_against(d["SegFormer-B3"], 6):
        gt = lut15[np.asarray(Image.open(f"{ATR_CACHE}/masks_orig/{int(k):06d}.png"))]
        image = np.asarray(Image.open(f"{ATR_CACHE}/images/{int(k):06d}.jpg").convert("RGB").resize((gt.shape[1], gt.shape[0])))
        rows.append([("ATR photo", image), ("labels", np.where(gt[..., None] == 255, 255, pal15[np.minimum(gt, 14)]).astype(np.uint8))]
                    + [(f"{n} ({d[n].image_score(k):.2f})", pal15[m.predict(image)]) for n, m in models.items()])
    show(rows, "YOLO26x-sem against SegFormer-B3 on ATR: the 6 photos where YOLO falls furthest behind",
         f"{DRIVE_ROOT}/reports/why_yolo_worst_atr.png")
    del models
    torch.cuda.empty_cache()


# ---- One section per model: why it scores lower than our final model, in numbers and in photos.
for name, (make, make_lut) in MODELS.items():
    if name == OURS:
        continue
    print()
    print("=" * 110)
    print(f"WHY {name.upper()} IS BELOW OUR FINAL MODEL ({len(clean)} CCP photos, none seen in training)")
    for line in explain(diags[name], diags[OURS], name, "ours", top=5):
        print(" -", line)
    exclude = ("shoes",) if name == "FASHN parser" else ()   # FASHN has no shoes class: not counted when choosing photos
    if exclude:
        print(" - FASHN has no shoes class (its 'feet' is a body class), so its shoe pixels count as skin: a different label definition, not a mistake.")
    model = make()
    lut = make_lut(model)
    rows = []
    for gap, k in diags[name].worst_against(diags[OURS], 6, exclude):
        image = load_image(by_id[k]["photo"])
        rows.append([(f"photo {k}", image), ("CCP labels", PAL_IGN[by_id[k]["gt"]]),
                     (f"ours ({diags[OURS].image_score(k, exclude):.2f})", PAL[ours_lut[ours_model.predict(image)]]),
                     (f"{name} ({diags[name].image_score(k, exclude):.2f})", PAL[lut[model.predict(image)]])])
    del model
    torch.cuda.empty_cache()
    show(rows, f"{name}: the 6 CCP photos where it falls furthest behind our final model (garment IoU per photo)",
         f"{DRIVE_ROOT}/reports/why_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_") + "_worst_ccp.png")
    if name.startswith("YOLO"):
        yolo_on_atr()
del ours_model
torch.cuda.empty_cache()

# ---- All models side by side: which kind of error each one makes.
print()
print("=" * 110)
print("ALL MODELS SIDE BY SIDE (CCP)")
print()
print("Outlines, insides and garment size: pixel accuracy in % and garment IoU by size")
rows = {n: {**{k: 100 * v for k, v in d.outline_rows().items()}, **d.size_rows()} for n, d in diags.items()}
print(pd.DataFrame(rows).round(3).to_string())
print()
print("Missed garments: % of photos with the garment where less than 20% of it was found")
print(pd.DataFrame({n: {r["garment"]: r["missed (%)"] for r in d.presence_rows()} for n, d in diags.items()}).round(1).to_string())
print()
print("Invented garments: % of photos without the garment where the model shows at least 0.2% of the photo as it")
print(pd.DataFrame({n: {r["garment"]: r["invented (% of photos without it)"] for r in d.presence_rows()} for n, d in diags.items()}).round(1).to_string())

## What to send back

Everything cells 6 and 7 print, and the pictures from cell 7. They are also saved on Drive in `clothseg/reports/` (`compare_models_ccp.csv`, `why_*.png`).